# LahjaLens — entraînement du sentiment avec Spark ML

Ce notebook doit être exécuté dans le conteneur `spark-master` avec `spark-submit`. Il utilise le cluster Spark, les splits AfriSenti préparés et une régression logistique multiclasses pondérée.

In [ ]:
from datetime import datetime, timezone
import json
import os
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import HashingTF, IDF, RegexTokenizer
from pyspark.sql import SparkSession, functions as F

INPUT_DIR = '/opt/lahjalens/data/processed/afrisenti/v1'
MODEL_DIR = 'hdfs://namenode:9000/data/lahjalens/models/afrisenti_v1'
spark = (SparkSession.builder.appName('LahjaLens-AfriSenti-Sentiment').master('spark://spark-master:7077').getOrCreate())
spark.sparkContext.setLogLevel('WARN')

## 1. Charger les splits préparés

Nous ne faisons pas de division aléatoire : AfriSenti fournit déjà les ensembles train/dev/test.

In [ ]:
def read_split(split):
    return (spark.read.options(header=True, mode='FAILFAST', quote=chr(34), escape=chr(34), multiLine=True).csv(f'{INPUT_DIR}/{split}.csv')
            .select(F.trim('comment_text').alias('comment_text'), F.col('label').cast('double').alias('label'), 'record_id', 'source_dataset')
            .where(F.col('comment_text').isNotNull() & (F.length('comment_text') > 0))
            .where(F.col('label').isin([0.0, 1.0, 2.0])))

train = read_split('train').cache()
dev = read_split('dev').cache()
test = read_split('test').cache()
train.groupBy('label').count().orderBy('label').show()

## 2. Corriger le déséquilibre avec des poids de classe

Les poids sont calculés uniquement sur train. Le dev et le test restent inchangés.

In [ ]:
counts = {int(row['label']): row['count'] for row in train.groupBy('label').count().collect()}
total = sum(counts.values())
weights = {label: total / (3.0 * count) for label, count in counts.items()}
mapping = F.create_map(*[item for label, weight in weights.items() for item in (F.lit(float(label)), F.lit(float(weight)))])
train = train.withColumn('class_weight', mapping[F.col('label')])
weights

## 3. Construire et entraîner le pipeline

Tokenizer → HashingTF → IDF → Logistic Regression multiclasses.

In [ ]:
tokenizer = RegexTokenizer(inputCol='comment_text', outputCol='tokens', pattern='\\s+', gaps=True, minTokenLength=1)
hashing = HashingTF(inputCol='tokens', outputCol='raw_features', numFeatures=1 << 18)
idf = IDF(inputCol='raw_features', outputCol='features', minDocFreq=2)
classifier = LogisticRegression(labelCol='label', featuresCol='features', weightCol='class_weight', maxIter=30, regParam=0.1, family='multinomial')
pipeline = Pipeline(stages=[tokenizer, hashing, idf, classifier])
model = pipeline.fit(train)
model.write().overwrite().save(MODEL_DIR)

## 4. Évaluer train, dev et test

Le score principal à comparer entre modèles est le macro-F1 calculé par classe. Spark nomme `f1` la moyenne pondérée ; nous affichons donc aussi les métriques par classe.

In [ ]:
def evaluate(frame, split):
    predictions = model.transform(frame).cache()
    result = {'split': split, 'rows': predictions.count()}
    for metric in ('accuracy', 'f1', 'weightedPrecision', 'weightedRecall'):
        result[metric] = float(MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName=metric).evaluate(predictions))
    result['per_class'] = {}
    for label, name in ((0.0, 'negative'), (1.0, 'neutral'), (2.0, 'positive')):
        result['per_class'][name] = {'recall': float(MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='recallByLabel', metricLabel=label).evaluate(predictions)), 'precision': float(MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='precisionByLabel', metricLabel=label).evaluate(predictions))}
    return result

metrics = {'created_at': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'), 'dataset': 'afrisenti', 'model': 'hashingTF_idf_logistic_regression', 'class_weights': weights, 'train': evaluate(train, 'train'), 'dev': evaluate(dev, 'dev'), 'test': evaluate(test, 'test')}
print(json.dumps(metrics, ensure_ascii=False, indent=2))
spark.sparkContext.parallelize([json.dumps(metrics, ensure_ascii=False)], 1).saveAsTextFile(f'{MODEL_DIR}/metrics')

In [ ]:
spark.stop()